# RIT Competition Prep — FI2 & EV1

Two cases, two completely different edges:

| | **FI2** Coupon Govt Bonds | **EV1** Equity Valuation |
|---|---|---|
| Uncertainty | **None.** Rates given in advance | EPS news arrives during the case |
| Edge | Price every security exactly, trade the noise | React to news faster than the room |
| Failure mode | Paying the 2c commission for a 1c edge | Churning a good position on noise |
| Key number | $0.02/bond commission | $0.01/share fee, ±100,000 share cap |

This notebook derives both fair values and connects to a live RIT session.

In [ ]:
import sys, pathlib
ROOT = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (d / "ritlib").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ritlib.pricing import *
from ritlib.news import parse_eps_news

plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## 1. FI2 — solving the case before it starts

The brief says it outright: *"traders can solve the exact value of the bonds at all times."*

Cash interest is credited **every 12 ticks**, 26 times per 312-tick period:

$$r_{week} = (1+r_{annual})^{1/52} - 1$$

Because the credit is discrete, fair value is a **step function** — flat between compounding
ticks. Check the weekly rates against the brief's own numbers (0.1302% / 0.1659%):

In [ ]:
print(f"Period 1 weekly rate: {weekly_rate(1)*100:.4f}%   (brief says 0.1302%)")
print(f"Period 2 weekly rate: {weekly_rate(2)*100:.4f}%   (brief says 0.1659%)")
print(f"\nPeriod 2 discount factor: {period2_full_discount():.6f}  = 1/1.09^0.5")

### Fair value paths

`bond_clean` is the number that matters: RIT quotes bonds clean, so that is what
you compare to the order book. You then pay clean + accrued.

In [ ]:
rows = []
for period in (1, 2):
    for tick in range(0, TICKS_PER_PERIOD + 1, 4):
        v = fi2_values(min(tick, TICKS_PER_PERIOD), period)
        rows.append({"period": period, "tick": tick, "TB6M": v.tb6m, "TB12M": v.tb12m,
                     "clean": v.bond_clean, "dirty": v.bond_dirty, "accrued": v.accrued})
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, p in zip(axes, (1, 2)):
    d = df[df.period == p]
    if p == 1:
        ax.plot(d.tick, d.TB6M, label="TB6M")
    ax.plot(d.tick, d.TB12M, label="TB12M")
    ax.plot(d.tick, d.clean, label="1YCP clean", lw=2)
    ax.plot(d.tick, d.dirty, label="1YCP dirty", ls="--", alpha=0.6)
    ax.set_title(f"Period {p}"); ax.set_xlabel("tick"); ax.legend(fontsize=8)
axes[0].set_ylabel("price")
plt.suptitle("FI2 theoretical values — the staircase is the weekly interest credit")
plt.tight_layout(); plt.show()

df[df.period == 1].head(8).round(4)

### The staircase matters

Zoom in: value is **flat for 12 ticks then jumps**. If you price continuously you will
think the bond is drifting cheap right before each compounding tick and pay up for nothing.
The step is worth ~0.13 of a cent per tick — small, but so is your edge.

In [ ]:
d = df[(df.period == 1) & (df.tick <= 60)]
plt.step(d.tick, d.TB6M, where="post", lw=2, label="discrete (correct)")
cont = [100 * (1 + weekly_rate(1)) ** -weeks_remaining(t, discrete=False) for t in d.tick]
plt.plot(d.tick, cont, ls="--", label="continuous (wrong)")
plt.axvline(12, color="grey", lw=0.8); plt.axvline(24, color="grey", lw=0.8)
plt.title("TB6M, first 60 ticks"); plt.xlabel("tick"); plt.legend(); plt.show()

### The model-free trade

The bond's cash flows are **exactly** replicated by bills:

$$\text{1YCP} = 0.05 \times \text{TB6M} + 1.05 \times \text{TB12M}$$

($5 = 0.05 \times \$100$ at the end of period 1, $105 = 1.05 \times \$100$ at the end of period 2.)

This holds **whatever the true discount rates are**. If the bond trades away from the basket
you are arbitraging, not forecasting — this is the trade to take if you ever doubt your rate
assumptions. Cost: 3 legs × $0.02 commission, and all three must fill.

In [ ]:
v = fi2_values(100, 1)
print(f"theo clean from discounting : {v.bond_clean:.6f}")
print(f"theo clean from replication : {replication_fair_bond(v.tb6m, v.tb12m, 100, 1):.6f}")

# What a mispriced basket looks like
bond_ask, tb6m_bid, tb12m_bid = 101.30, v.tb6m + 0.02, v.tb12m + 0.03
basket = replication_fair_bond(tb6m_bid, tb12m_bid, 100, 1)
edge = basket - bond_ask - 0.02 * 2.1
print(f"\nbuy bond @{bond_ask}, sell basket @{basket:.3f}  ->  {edge:+.3f}/bond after commission")
print(f"at 1,000 bonds (max order size): ${edge*1000:,.0f}")

## 2. EV1 — fair value is a formula, the edge is speed

$$P_{final} = (Q_1 + Q_2 + Q_3 + Q_4) \times 12.5$$

Starting estimates sum to 1.24 → **$15.50**. Last year's actuals summed to 0.95 → $11.875,
which is the anchor the slow traders will drift toward. Ignore it.

Every quarter's EPS surprise moves fair value by `surprise × 12.5`:

In [ ]:
book = EPSBook()
print("start:", book.summary())

for surprise in (0.02, 0.05, 0.10):
    print(f"  a {surprise:+.2f} EPS surprise moves FV by ${surprise*COMP_PE:+.2f} "
          f"= ${surprise*COMP_PE*EV1_POSITION_LIMIT:+,.0f} at full size")

scenarios = {"all estimates hit": {1:.40,2:.24,3:.27,4:.33},
             "10% beat":          {1:.44,2:.26,3:.30,4:.36},
             "10% miss":          {1:.36,2:.22,3:.24,4:.30}}
for name, eps in scenarios.items():
    print(f"\n{name:20} sum={sum(eps.values()):.2f}  FV=${sum(eps.values())*COMP_PE:.2f}")

### Position sizing is the whole game

The cap is ±100,000 shares and the fee is $0.01/share. A round trip costs $0.02/share —
**2,000 dollars** at full size. That is why the bot holds rather than churning: PI settles
at fair value, so a position bought below FV is already a winner. You exit when the price
overshoots to the *other* side, not when it merely returns to fair.

In [ ]:
fv = 15.50
prices = np.linspace(14.5, 16.5, 200)
full_edge, max_pos, fee = 0.25, 100_000, 0.01

def target(price):
    if price < fv - fee - 0.03: return min(1.0, (fv - fee - price)/full_edge) * max_pos
    if price > fv + fee + 0.03: return -min(1.0, (price - fv - fee)/full_edge) * max_pos
    return 0.0

plt.plot(prices, [target(p) for p in prices], lw=2)
plt.axvline(fv, color="k", ls="--", lw=1, label="fair value")
plt.axhspan(-max_pos, max_pos, alpha=0.05, color="green")
plt.fill_between([fv-0.04, fv+0.04], -max_pos, max_pos, alpha=0.15, color="grey")
plt.title("EV1 target position vs price (grey = dead band, hold existing position)")
plt.xlabel("PI price"); plt.ylabel("target shares"); plt.legend(); plt.show()

### News parsing sanity check

The bot reads `/v1/news` and pulls EPS out of the text. Verify it handles the shapes you
expect — and remember the escape hatch: if a headline defeats the regex mid-case, write
`eps_override.json` instead of debugging under time pressure.

In [ ]:
samples = [
    ("Prandium Industries announces Q1 earnings", "Reported EPS of $0.42 for the first quarter."),
    ("Analysts revise Q2 estimates", "Analysts now estimate Q2 EPS of 0.28, down from 0.24."),
    ("PI Q3 earnings release", "Q3 EPS came in at $0.19, missing the consensus of 0.27."),
    ("Prandium wins new contract", "No financial details were disclosed."),
]
for h, b in samples:
    print(f"{h:45} -> {parse_eps_news(h, b)}")

## 3. Microstructure — how price actually moves in RIT

Fair value tells you *what* to trade. Microstructure tells you *how much* and *at what
price you will really get filled*. Three facts from Rotman's own tutorials drive
everything in `ritlib/microstructure.py`:

1. **Market orders walk the book.** Your fill is a VWAP across price levels, not the
   touch price on screen.
2. **Liquidity = spread AND depth** — "how close the bids and asks are, and the volume
   available at each price level".
3. **ANON** is the computer participant. You cannot tell an informed ANON from an
   uninformed one by its label.

### Validating against Rotman's worked example

Their tutorial states that a 5,000-share market buy of TAME fills 700 @ 25.54,
1500 @ 25.55, 2100 @ 25.63, 700 @ 25.74. Our `walk()` must reproduce that exactly.

In [ ]:
from ritlib.microstructure import OrderBook, Tape, plan_execution

payload = {"bids": [{"price": 25.20, "quantity": 800,  "trader_id": "ANON"},
                    {"price": 24.79, "quantity": 800,  "trader_id": "ANON"},
                    {"price": 24.75, "quantity": 5000, "trader_id": "Kevin"}],
           "asks": [{"price": 25.54, "quantity": 700,  "trader_id": "ANON"},
                    {"price": 25.55, "quantity": 1500, "trader_id": "ANON"},
                    {"price": 25.63, "quantity": 2100, "trader_id": "ANON"},
                    {"price": 25.74, "quantity": 900,  "trader_id": "ANON"},
                    {"price": 25.80, "quantity": 3400, "trader_id": "Kevin"}]}
book = OrderBook.from_api(payload, "TAME")

est = book.walk("BUY", 5000)
rotman = (700*25.54 + 1500*25.55 + 2100*25.63 + 700*25.74) / 5000
print(f"our walk(): {est}")
print(f"VWAP {est.vwap:.4f} vs Rotman's {rotman:.4f} -> {'MATCH' if abs(est.vwap-rotman)<1e-9 else 'MISMATCH'}")
print(f"\nScreen says {book.best_ask:.2f}. You actually pay {est.vwap:.4f}.")
print(f"That {est.slippage:.4f}/share gap is {est.slippage*5000:,.0f} dollars on this order.")

### The cost of size

Slippage is not linear — it steps as each level is exhausted. This curve is why
"take the whole edge" is usually wrong: past a point you are paying more in impact
than the mispricing is worth.

In [ ]:
sizes = np.arange(100, 8600, 100)
vwaps = [book.walk("BUY", int(q)).vwap for q in sizes]
filled = [book.walk("BUY", int(q)).filled for q in sizes]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(sizes, vwaps, lw=2)
ax1.axhline(book.best_ask, ls="--", color="grey", label=f"touch {book.best_ask}")
ax1.set_title("Average fill price vs order size"); ax1.set_xlabel("shares")
ax1.set_ylabel("VWAP"); ax1.legend()

ax2.plot(sizes, np.array(filled)/sizes, lw=2, color="firebrick")
ax2.set_title("Fill ratio — where the book runs out"); ax2.set_xlabel("shares")
ax2.set_ylabel("filled / requested"); ax2.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

### Sizing the right way

`max_qty_for_avg_price` answers the only question that matters when you have found a
mispricing: **how much of it is actually there?**

Say fair value is 25.70 and costs are 1c. Any average fill below 25.69 makes money.

In [ ]:
fair, fee = 25.70, 0.01
breakeven = fair - fee

naive = book.qty_at_or_better("BUY", breakeven)       # only levels priced under
smart = book.max_qty_for_avg_price("BUY", breakeven)  # blended average under

print(f"fair {fair}  breakeven avg {breakeven}")
print(f"  levels strictly better than breakeven : {naive:,} shares")
print(f"  largest order whose VWAP still clears : {smart:,} shares")
e = book.walk('BUY', smart)
print(f"  -> vwap {e.vwap:.4f} (<= {breakeven}), profit ${(fair - e.vwap - fee) * smart:,.0f}")
print(f"\nTaking one more share: vwap {book.walk('BUY', smart+1).vwap:.6f} — over the line.")
print(f"\nsliced into legal orders: {plan_execution(book, 'BUY', smart, breakeven, 10_000)}")

### Imbalance, microprice and who is on the other side

- **Microprice** weights the mid by size. A huge bid against a thin ask means the next
  print is at the ask — so value sits above the arithmetic mid.
- **Imbalance** in [-1, +1] is a short-horizon pressure gauge. Use it for *timing*, not
  for valuation.
- **ANON share** tells you whether you are trading against computers (noise, safe) or
  named humans competing for the same edge.

In [ ]:
for bid_sz, ask_sz in [(500, 500), (10000, 500), (500, 10000)]:
    b = OrderBook.from_api({"bids": [{"price": 9.99, "quantity": bid_sz}],
                            "asks": [{"price": 10.01, "quantity": ask_sz}]}, "X")
    print(f"bid {bid_sz:>6} x ask {ask_sz:>6} | mid {b.mid:.4f} | "
          f"micro {b.microprice:.4f} | imbalance {b.imbalance():+.2f}")

print(f"\nTAME book: ANON holds {book.anon_share('ASK'):.0%} of the ask side, "
      f"{book.anon_share('BID'):.0%} of the bid side")
print(f"liquidity score for a 5,000 order: {book.liquidity_score(5000)}")

### Reading the tape

`signed_volume` applies the tick rule: a print above the previous one is buyer-initiated,
below is seller-initiated. Persistent one-sided flow is the footprint of someone working
a large order — and the thing most likely to run over a resting quote.

In [ ]:
tape = Tape.from_api([{"id":i,"tick":i,"price":p,"quantity":q} for i,(p,q) in enumerate(
    [(25.50,100),(25.55,300),(25.55,200),(25.60,500),(25.62,400),(25.58,150)], 1)])
print(f"volume {tape.volume}  vwap {tape.vwap:.4f}  last {tape.last_price}")
print(f"signed volume {tape.signed_volume():+}  flow ratio {tape.flow_ratio():+.3f}")
print(f"realized vol (per tick) {tape.realized_vol():.5f}")
print("\nflow ratio near +1 = relentless buying; near 0 = two-way, safe to quote both sides")

## 4. Connect to a live session

Start the mock server first (`python mock/mock_rit_server.py --case fi2`), or point this at
the real RIT client on competition day.

In [ ]:
from ritlib.client import RITClient, RITError

c = RITClient()          # http://localhost:9999/v1, key from $RIT_API_KEY
try:
    print(c.case())
    print(pd.DataFrame(c.securities())[["ticker","bid","ask","last","position"]])
except RITError as e:
    print("not connected:", e)
    print("start the mock: python mock/mock_rit_server.py --case fi2")

In [ ]:
# Live edge table — re-run this cell repeatedly during the case
from IPython.display import display

try:
    case = c.case()
    v = fi2_values(min(case["tick"], TICKS_PER_PERIOD), case["period"])
    out = []
    for s in c.securities():
        theo = v.theo(s["ticker"])
        if theo is None: continue
        out.append({"ticker": s["ticker"], "theo": round(theo,4),
                    "bid": s["bid"], "ask": s["ask"],
                    "buy_edge": round(theo - s["ask"] - BOND_COMMISSION, 4),
                    "sell_edge": round(s["bid"] - theo - BOND_COMMISSION, 4),
                    "pos": s["position"]})
    display(pd.DataFrame(out).style.background_gradient(
        cmap="RdYlGn", subset=["buy_edge","sell_edge"], vmin=-0.05, vmax=0.05))
except RITError as e:
    print("not connected:", e)